In [1]:
from pyomo.dae import ContinuousSet, DerivativeVar
import pyomo.environ as pyo
from pyomo.contrib.doe import DesignOfExperiments
import numpy as np


class scalarPDE:
    """
    1D diffusion PDE with:
    - left boundary heat flux control u(t)
    - right boundary insulated (no flux)
    - uniform initial temperature T0
    Structured so DesignOfExperiments can call get_labeled_model().
    """

    def __init__(self, data, nfe_t, ncp_t, nfe_x, ncp_x):
        self.data   = data
        self.nfe_t  = nfe_t
        self.ncp_t  = ncp_t
        self.nfe_x  = nfe_x
        self.ncp_x  = ncp_x
        self.model  = None

    def get_labeled_model(self):
        if self.model is None:
            self._create_model()
            self._finalize_model()
            self._label_experiment()
        return self.model

    def _create_model(self):
        m = self.model = pyo.ConcreteModel()

        t_lo = min(self.data["t_range"])
        t_hi = max(self.data["t_range"])

        m.t = ContinuousSet(bounds=[t_lo, t_hi])
        m.x_space = ContinuousSet(bounds=[0.0, 1.0])

        m.T = pyo.Var(m.x_space, m.t, within=pyo.Reals)
        m.u = pyo.Var(m.t, within=pyo.Reals)

        m.alpha = pyo.Var(within=pyo.Reals)   # unknown parameter

        m.h   = pyo.Param(initialize=self.data["h"],  mutable=False)
        m.T_c = pyo.Param(initialize=self.data["Tc"], mutable=False)

        m.T0 = pyo.Var(initialize=self.data["T0"], within=pyo.Reals)

        m.dTdt   = DerivativeVar(m.T, wrt=m.t)
        m.dTdx   = DerivativeVar(m.T, wrt=m.x_space)
        m.d2Tdx2 = DerivativeVar(m.dTdx, wrt=m.x_space)

        @m.Constraint(m.x_space, m.t)
        def pde_eqn(m, x, t):
            xL = m.x_space.first()
            xR = m.x_space.last()
            if (x == xL) or (x == xR):
                return pyo.Constraint.Skip
            return m.dTdt[x, t] == m.alpha * m.d2Tdx2[x, t]

    def _finalize_model(self):
        m = self.model

        control_points = self.data["control_points"]
        t_range        = self.data["t_range"]
        u_bounds       = self.data["u_bounds"]
        alpha_nominal  = self.data["alpha_nominal"]
        x_range        = self.data.get("x_range", None)
        x_sensor_val   = self.data.get("x_sensor", 0.5)

        if x_range is not None:
            for xv in x_range:
                m.x_space.update([xv])
        if x_sensor_val not in list(m.x_space):
            m.x_space.update([x_sensor_val])

        x_left_val  = min(list(m.x_space))
        x_right_val = max(list(m.x_space))

        for tt in t_range:
            m.t.update([tt])
        for tt in control_points.keys():
            m.t.update([tt])

        m.t_control = pyo.Set(initialize=list(control_points.keys()), ordered=True)

        # --- alpha is free (unknown parameter)
        m.alpha.setlb(1e-8)
        m.alpha.setub(10.0)
        m.alpha.value = alpha_nominal

        # --- fix design variables so base model is square ---
        umin, umax = u_bounds
        for t in m.t:
            m.u[t].setlb(umin)
            m.u[t].setub(umax)
            prev_ctrl_times = [tc for tc in control_points if tc <= t]
            if len(prev_ctrl_times) == 0:
                first_tc = min(control_points.keys())
                m.u[t].value = control_points[first_tc]
            else:
                last_tc = max(prev_ctrl_times)
                m.u[t].value = control_points[last_tc]
            m.u[t].fix(m.u[t].value)  # <-- FIXED

        m.T0.fix(pyo.value(m.T0))  # <-- FIXED

        @m.Constraint(m.t - m.t_control)
        def u_hold(m, t):
            prev_tc = max(tc for tc in m.t_control if tc < t)
            return m.u[t] == m.u[prev_tc]

        @m.Constraint(m.t)
        def left_bc(m, t):
            return m.dTdx[x_left_val, t] == -m.u[t]

        @m.Constraint(m.t)
        def right_bc(m, t):
            return m.dTdx[x_right_val, t] == 0# -(m.h*m.T[x_right_val,t])

        m.x_sensor     = pyo.Param(initialize=x_sensor_val,  mutable=False)
        m.x_left_node  = pyo.Param(initialize=x_left_val,    mutable=False)
        m.x_right_node = pyo.Param(initialize=x_right_val,   mutable=False)

        disc_space = pyo.TransformationFactory("dae.collocation")
        disc_space.apply_to(m, wrt=m.x_space, nfe=self.nfe_x, ncp=self.ncp_x)

        disc_time = pyo.TransformationFactory("dae.collocation")
        disc_time.apply_to(m, wrt=m.t, nfe=self.nfe_t, ncp=self.ncp_t)

        t0 = min(list(m.t))
        def ic_rule(m, x):
            return m.T[x, t0] == m.T0
        m.ic = pyo.Constraint(m.x_space, rule=ic_rule)

        T0_lo, T0_hi = self.data.get("T0_bounds", (0.1, 2.0))
        m.T0.setlb(T0_lo)
        m.T0.setub(T0_hi)

        # Bound the temperature field everywhere to a reasonable range
        # (adjust to your units/physics as needed)
        m.T.setlb(T0_lo - 5.0)   # e.g., allow some deviation around initial temp
        m.T.setub(T0_hi + 5.0)

    def _label_experiment(self):
        m = self.model

        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        x_s = pyo.value(m.x_sensor)
        for t_m in m.t_control:
            m.experiment_outputs[m.T[x_s, t_m]] = None

        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        meas_err = 0
        for t_m in m.t_control:
            m.measurement_error[m.T[x_s, t_m]] = meas_err

        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_inputs[m.T0] = None
        for t_ctrl in m.t_control:
            m.experiment_inputs[m.u[t_ctrl]] = None

        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters[m.alpha] = pyo.value(m.alpha)



# --------------------------
# data for the PDE experiment
# --------------------------

# make ALL control points 0.0 to avoid forcing gradients at t≈0
flat_u = {
    0.0:   0.5,
    0.125: 0.5,
    0.25:  0.5,
    0.375: 0.5,
    0.5:   0,
    0.625: -0.5,
    0.75:  -0.2,
    0.875: -0.5,
    1.0:   -0.5,
}

data_pde = {
    "x_range":   [0.0, 0.25, 0.5, 0.75, 1.0],
    "x_sensor":  0.5,
    "t_range":   [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0],

    "control_points": flat_u,
    "u_bounds": (-0.6, 0.6),

    "T0": 0.5,
    "T0_bounds": (0.5, 2.0),

    "h": 1,
    "Tc": 0,
    "alpha_nominal": 0.5,
}

# discretization
nfe_x = 8
ncp_x = 3
nfe_t = 8
ncp_t = 3


# wrapper class to satisfy DesignOfExperiments API
class ExperimentWrapper:
    def __init__(self, exp_obj):
        self._exp = exp_obj
    def get_labeled_model(self):
        return self._exp.get_labeled_model()


# build wrapped experiment
pde_expt = scalarPDE(
    data=data_pde,
    nfe_t=nfe_t,
    ncp_t=ncp_t,
    nfe_x=nfe_x,
    ncp_x=ncp_x,
)
wrapped_expt = ExperimentWrapper(pde_expt)

# DoE setup
fd_formula = "central"
step_size = 1e-3
objective_option = "determinant"
scale_nominal_param_value = True

doe_obj = DesignOfExperiments(
    wrapped_expt,
    fd_formula=fd_formula,
    step=step_size,
    objective_option=objective_option,
    gradient_method = "pynumero",
    scale_constant_value=1,
    scale_nominal_param_value=scale_nominal_param_value,
    prior_FIM=None,
    jac_initial=None,
    fim_initial=None,
    L_diagonal_lower_bound=1e-7,
    solver=pyo.SolverFactory("ipopt"),
    tee=False,
    get_labeled_model_args=None,
    _Cholesky_option=True,
    _only_compute_fim_lower=True,
)

# run DoE
doe_obj.run_doe()

# report results
print("Optimal experiment values:")
opt_design = doe_obj.results["Experiment Design"]

if len(opt_design) > 0:
    print(f"\tInitial temperature T0*: {opt_design[0]:.4f}")

if len(opt_design) > 1:
    u_list = opt_design[1:]
    print(
        "\tControl inputs u(t_ctrl)*: ["
        + ", ".join(f"{val:.4f}" for val in u_list)
        + "]"
    )

print("FIM at optimal design:")
print(np.array(doe_obj.results["FIM"]))

print("Objective value at optimal design:",
      pyo.value(doe_obj.model.objective))

print("Experiment Design Names:", doe_obj.results["Experiment Design Names"])
print("Solve time (s):", doe_obj.results["Solve Time"])
print("Build time (s):", doe_obj.results["Build Time"])
print("Initialization time (s):", doe_obj.results["Initialization Time"])
print("Total wall time (s):", doe_obj.results["Wall-clock Time"])


ValueError: The model is not square: the number of variables minus unknown parameters does not equal the number of constraints.
This is required for the (automatic) differentiation to work correctly.